# Distilling learned reasoning: 1.5B GRPO → 0.5B
### Does the RL gain travel as text into a smaller model?

| § | Question | Figure |
|---|---|---|
| 1 | **The teacher, re-measured** — base 1.5B vs. GRPO step 400 on this run's instrument | baseline table |
| 2 | **Harvest** — correct traces on GSM8K train from three sources | coverage, trace length |
| 3 | **Matched SFT** — Qwen2.5-0.5B-Instruct on each source, three seeds | training loss |
| 4 | **Does the gain travel?** | pass@1 per arm · paired gains · transfer ratio |
| 5 | **Both axes for the student** | maj@k, accuracy vs. budget, trace length |
| 6 | **One problem, three models** | a random test problem |

- **Three trace sources, one student, matched data.** The 0.5B's own traces (a rejection-sampling control), base 1.5B's, and GRPO 1.5B's. Each source contributes one correct trace per problem on the *same* problems, so the arms differ only in whose reasoning the student reads.
- **Transfer = (0.5B on GRPO traces − 0.5B on base-1.5B traces) ÷ (GRPO teacher − base teacher).** Both differences are paired over the same 600 test problems, with a 95% bootstrap interval.
- **A trace is kept if it stopped on its own and the teacher's grader marks it correct.** Same `extract` the teacher was rewarded with; no format requirement.
- **Plain SFT, loss on the trace only.** No reward, no teacher logits, fp32 master weights.
- **Every run is full and re-measures its own teachers.** A disconnect resumes the unfinished run; a finished run (one with `derived.json`) is never read again.

**Time on an A100 80GB:** ≈ 1.5–2 h (harvest ≈ 30 min, 12 evals ≈ 45 min, 9 SFTs ≈ 35 min). Any GPU with ≥ 24 GB works, just slower.

> **Run order.** Cell 0.1 installs and then **restarts the runtime on purpose**. Afterwards run all cells from the top; the install cell skips in a second.

## 0. Environment

### 0.1 Install (idempotent; restarts the runtime once)
Unchanged from the GRPO notebook.

In [ ]:
import os, sys, subprocess, importlib.metadata as md
MARK = "/content/.two_axis_env_ok"

def pip(*args):
    print(">", "pip", *args); subprocess.run([sys.executable, "-m", "pip", *args], check=True)

if not os.path.exists(MARK):
    pip("install", "-q", "--upgrade", "pip")
    pip("uninstall", "-y", "-q", "torchvision", "torchaudio")        # Colab's copies will mismatch vLLM's torch; nothing here needs them
    pip("install", "-q", "--upgrade", "vllm>=0.8.5,<0.33.0")         # brings its own torch + nvidia-* libs, all mutually consistent
    pip("install", "-q", "--upgrade", "trl>=0.19", "peft", "datasets", "accelerate", "pandas", "matplotlib")
    # --- self-check in a fresh interpreter; repair the one classic failure (NCCL wheel not matching torch) ---
    chk = subprocess.run([sys.executable, "-c", "import torch, vllm, trl; print(torch.__version__, vllm.__version__, trl.__version__)"],
                         capture_output=True, text=True)
    if chk.returncode != 0 and "nccl" in chk.stderr.lower():
        need = [r for r in md.requires("torch") if r.startswith("nvidia-nccl")][0].split(";")[0].strip()
        print("repairing NCCL ->", need); pip("install", "-q", "--force-reinstall", "--no-deps", need)
        chk = subprocess.run([sys.executable, "-c", "import torch, vllm, trl"], capture_output=True, text=True)
    if chk.returncode != 0:
        print(chk.stderr[-3000:]); raise SystemExit("environment check failed — see error above")
    print("versions:", chk.stdout.strip())
    open(MARK, "w").write("ok")
    print("\nEnvironment ready. Restarting the runtime now (expected). Then run all cells from the top.")
    os.kill(os.getpid(), 9)
else:
    print("environment already prepared — skipping install")

### 0.2 Imports and GPU

In [ ]:
import os, sys, json, gc, re, shlex, shutil, hashlib, subprocess, random, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import trl, vllm, transformers

assert torch.cuda.is_available()
gpu = torch.cuda.get_device_properties(0); VRAM_GB = round(gpu.total_memory / 2**30)
print(f"GPU: {gpu.name} | VRAM {VRAM_GB} GB | torch {torch.__version__} | vllm {vllm.__version__} | trl {trl.__version__} | transformers {transformers.__version__}")
assert VRAM_GB >= 22, f"{VRAM_GB} GB: SFT of the 0.5B at batch 8 x 1,344 tokens needs ~20 GB"
# optional: from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### 0.3 Knobs — the only cell you edit
`TEACHER_WEIGHTS` is a folder, a `.zip`/`.tar.gz` of one, a Hugging Face repo id, or `"latest-full-ft"`: the final 1.5B checkpoint of the latest finished run of the full fine-tuning notebook (`page_13_grpo_full_finetune.ipynb`).

In [ ]:
SEED         = 42                 # the GRPO notebook's seed: same 600 test problems, same eval sampling
MOUNT_DRIVE  = True
RESULTS_ROOT = "/content/drive/MyDrive/two_axis" if MOUNT_DRIVE else "/content/two_axis"

TEACHER_WEIGHTS = "latest-full-ft"     # the v3 bf16 checkpoint was f"{RESULTS_ROOT}/1.5B_A100-80GB_seed42/ckpt/step400"
TEACHER_BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"     # what GRPO started from; also the teacher's tokenizer
STUDENT_ID      = "Qwen/Qwen2.5-0.5B-Instruct"

# harvest: GSM8K train, every source sampled identically
GEN_K, GEN_TEMPERATURE, GEN_MAX_TOKENS = 8, 0.7, 1024
# SFT: full fine-tune of the student, effective batch SFT_BSZ * SFT_ACCUM
SFT_SEEDS, SFT_EPOCHS, SFT_LR, SFT_BSZ, SFT_ACCUM = [42, 43, 44], 3, 1e-5, 8, 4
# eval: the GRPO notebook's sweep, unchanged
N_EVAL, K_EVAL, EVAL_TEMPERATURE, EVAL_MAX_TOKENS = 600, 32, 0.7, 1024
BUDGETS = [64, 128, 256, 512, 1024]
KEEP_ON_DRIVE = ["grpo15"]        # sources whose trained students are copied to Drive (≈1 GB each); the rest stay on local disk

KNOBS = dict(SEED=SEED, TEACHER_BASE_ID=TEACHER_BASE_ID, STUDENT_ID=STUDENT_ID, GEN_K=GEN_K, GEN_TEMPERATURE=GEN_TEMPERATURE,
             GEN_MAX_TOKENS=GEN_MAX_TOKENS, SFT_SEEDS=SFT_SEEDS, SFT_EPOCHS=SFT_EPOCHS, SFT_LR=SFT_LR, SFT_BSZ=SFT_BSZ,
             SFT_ACCUM=SFT_ACCUM, N_EVAL=N_EVAL, K_EVAL=K_EVAL, EVAL_TEMPERATURE=EVAL_TEMPERATURE,
             EVAL_MAX_TOKENS=EVAL_MAX_TOKENS, BUDGETS=BUDGETS)

if MOUNT_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
ROOT = Path(RESULTS_ROOT) / "distill_1.5B-grpo_to_0.5B"; ROOT.mkdir(parents=True, exist_ok=True)
unfinished = sorted(p for p in ROOT.glob("run-*") if not (p / "derived.json").exists())
if unfinished:
    RUN = unfinished[-1]; saved = json.load(open(RUN / "config.json"))
    changed = {k: (saved.get(k), v) for k, v in KNOBS.items() if saved.get(k) != v}
    assert not changed, f"{RUN.name} is unfinished but was started with other knobs {changed}: restore them, or rename that folder to start a new run"
    print(f"resuming unfinished {RUN}")
else:
    RUN = ROOT / f"run-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"; RUN.mkdir()
    json.dump(KNOBS, open(RUN / "config.json", "w"), indent=1); print(f"new run {RUN}")
LOCAL = Path("/content/distill_local") / RUN.name; LOCAL.mkdir(parents=True, exist_ok=True)   # students + trainer scratch
for d in ("eval", "harvest", "data", "logs", "sft_logs", "students"): (RUN / d).mkdir(exist_ok=True)

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed); os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    transformers.set_seed(seed)
seed_everything(SEED)

### 0.4 Shared module and worker scripts
`common.py` is the GRPO notebook's, verbatim: the student is trained and graded with the prompt and extractor the teacher was rewarded with. Three workers run as subprocesses, so no GPU memory leaks between stages.

In [ ]:
%%writefile common.py
import re, json
from collections import Counter
from datasets import load_dataset

SYSTEM = "Think step by step. Then give the final answer on its own line as: #### <number>"
NUM = r"-?\$?\d[\d,]*\.?\d*"

def normalize(s):
    if s is None: return None
    s = s.replace(",", "").replace("$", "").strip().rstrip(".")
    try:
        f = float(s); return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return None

def extract(text):
    """Last '#### n' if present, else the last number in the text (what a truncated trace yields)."""
    m = re.findall(r"####\s*(" + NUM + ")", text)
    if m: return normalize(m[-1])
    nums = re.findall(NUM, text)
    return normalize(nums[-1]) if nums else None

def gold(answer_field): return normalize(answer_field.split("####")[-1])

def to_prompt(ex):
    return {"prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["question"]}],
            "answer": gold(ex["answer"])}

def load_gsm8k(split, n=None, seed=0):
    ds = load_dataset("openai/gsm8k", "main", split=split).map(to_prompt).shuffle(seed=seed)
    return ds.select(range(n)) if n else ds

def maj(answers):
    c = Counter(a for a in answers if a is not None)
    return c.most_common(1)[0][0] if c else None

def ks_up_to(k_max):
    ks, k = [], 1
    while k <= k_max: ks.append(k); k *= 2
    return ks

def make_config(cls, **kw):
    """Build a TrainingArguments-derived config, dropping kwargs the installed transformers/trl no longer accept
    (e.g. transformers>=5.2 removed warmup_ratio). Prints what it dropped so nothing disappears silently."""
    import dataclasses
    valid = {f.name for f in dataclasses.fields(cls)}
    dropped = {k: v for k, v in kw.items() if k not in valid}
    if dropped: print(f"[compat] {cls.__name__} does not accept {dropped} in this version - dropped", flush=True)
    return cls(**{k: v for k, v in kw.items() if k in valid})

def completion_text(c):
    """TRL passes completions as plain strings (standard datasets) or as lists of chat messages (conversational)."""
    if isinstance(c, str): return c
    if isinstance(c, dict): return c.get("content", "")
    return c[-1]["content"] if c else ""

def reward_correct(completions, answer, **kw):
    return [1.0 if extract(completion_text(c)) == a else 0.0 for c, a in zip(completions, answer)]

def reward_format(completions, **kw):
    return [0.2 if re.search(r"####\s*" + NUM, completion_text(c)) else 0.0 for c in completions]

In [ ]:
%%writefile gen_traces.py
# Harvest worker: k samples per GSM8K train problem from one model, each graded with common.extract.
# Texts are kept only for samples that are correct AND stopped on their own (the SFT pool).
import argparse, json, os, time
from vllm import LLM, SamplingParams
from common import load_gsm8k, extract
p = argparse.ArgumentParser()
for f in ["--model", "--tokenizer", "--out"]: p.add_argument(f, required=True)
for f in ["--k", "--max_tokens", "--seed"]: p.add_argument(f, type=int, required=True)
p.add_argument("--temperature", type=float, required=True); p.add_argument("--gpu_util", type=float, default=0.85)
a = p.parse_args(); t0 = time.time()
ds = load_gsm8k("train", seed=a.seed)
llm = LLM(model=a.model, tokenizer=a.tokenizer, dtype="bfloat16", seed=a.seed, gpu_memory_utilization=a.gpu_util,
          max_model_len=512 + a.max_tokens, enable_prefix_caching=True)
tok = llm.get_tokenizer()
prompts = [tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True) for ex in ds]
outs = llm.generate(prompts, SamplingParams(n=a.k, temperature=a.temperature, top_p=0.95, max_tokens=a.max_tokens, seed=a.seed))
rows = []
for ex, out in zip(ds, outs):
    smp = []
    for s in out.outputs:
        ans = extract(s.text); ok = ans == ex["answer"]; stopped = s.finish_reason == "stop"
        smp.append({"answer": ans, "correct": ok, "stopped": stopped, "n_tokens": len(s.token_ids), "text": s.text if ok and stopped else None})
    rows.append({"question": ex["question"], "gold": ex["answer"], "samples": smp})
res = {"model": a.model, "k": a.k, "seconds": round(time.time() - t0, 1), "rows": rows}
json.dump(res, open(a.out + ".tmp", "w")); os.replace(a.out + ".tmp", a.out)
n_ok = sum(x["correct"] for r in rows for x in r["samples"]); n = sum(len(r["samples"]) for r in rows)
print(f"harvest done: {len(rows)} problems x {a.k} | correct {n_ok/n:.1%} | {res['seconds']/60:.1f} min")

In [ ]:
%%writefile sft_student.py
# SFT worker: full fine-tune of the student on one source's traces. Prompt-completion data -> loss on the trace only.
import argparse, json, os, sys, time, shutil
from pathlib import Path
p = argparse.ArgumentParser()
for f in ["--data", "--student", "--out"]: p.add_argument(f, required=True)
for f in ["--seed", "--bsz", "--accum", "--max_len"]: p.add_argument(f, type=int, required=True)
p.add_argument("--epochs", type=float, required=True); p.add_argument("--lr", type=float, required=True)
a = p.parse_args()
if (Path(a.out) / "config.json").exists(): print("student exists - skipping"); sys.exit(0)
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer
from common import make_config
set_seed(a.seed); t0 = time.time()
ds = Dataset.from_list(json.load(open(a.data))).shuffle(seed=a.seed)
steps = int(a.epochs * -(-len(ds) // (a.bsz * a.accum)))
# fp32 master weights under bf16 autocast: a 1e-5 Adam step is below bf16's resolution on most weights
model = AutoModelForCausalLM.from_pretrained(a.student, dtype=torch.float32)
tok = AutoTokenizer.from_pretrained(a.student)
args = make_config(SFTConfig, output_dir=a.out + "_trainer", seed=a.seed, data_seed=a.seed,
    num_train_epochs=a.epochs, learning_rate=a.lr, lr_scheduler_type="cosine", warmup_steps=max(1, int(0.05 * steps)),
    per_device_train_batch_size=a.bsz, gradient_accumulation_steps=a.accum, max_length=a.max_len,
    completion_only_loss=True, packing=False, bf16=True, gradient_checkpointing=True,
    logging_steps=10, save_strategy="no", report_to="none")
print(f"training on {len(ds)} traces | {steps} steps", flush=True)
t = SFTTrainer(model=model, args=args, train_dataset=ds, processing_class=tok)
t.train()
tmp = a.out + ".tmp"; shutil.rmtree(tmp, ignore_errors=True)
t.model.to(torch.bfloat16).save_pretrained(tmp, safe_serialization=True); tok.save_pretrained(tmp)
json.dump(t.state.log_history, open(f"{tmp}/log_history.json", "w"))
os.replace(tmp, a.out)     # a half-written student is never mistaken for a finished one
print(f"distill done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
%%writefile eval_arm.py
# Eval worker: the GRPO notebook's eval_ckpt.py, measuring the same things; it also keeps each problem's
# answers (for paired comparisons across arms) and its first sample's text.
import argparse, json, os, time
from vllm import LLM, SamplingParams
from common import load_gsm8k, extract, maj, ks_up_to
p = argparse.ArgumentParser()
for f in ["--model", "--tokenizer", "--out", "--tag", "--budgets"]: p.add_argument(f, required=True)
for f in ["--n", "--k", "--max_tokens", "--seed"]: p.add_argument(f, type=int, required=True)
p.add_argument("--temperature", type=float, required=True); p.add_argument("--gpu_util", type=float, default=0.85)
a = p.parse_args(); t0 = time.time()
ds = load_gsm8k("test", n=a.n, seed=a.seed)
llm = LLM(model=a.model, tokenizer=a.tokenizer, dtype="bfloat16", seed=a.seed, gpu_memory_utilization=a.gpu_util,
          max_model_len=512 + a.max_tokens, enable_prefix_caching=True)
tok = llm.get_tokenizer()
prompts = [tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True) for ex in ds]
outs = llm.generate(prompts, SamplingParams(n=a.k, temperature=a.temperature, top_p=0.95, max_tokens=a.max_tokens, seed=a.seed))
budgets = [int(b) for b in a.budgets.split(",")]; ks = ks_up_to(a.k)
per, lens, trunc = [], [], 0
for ex, out in zip(ds, outs):
    ss = out.outputs; L = [len(s.token_ids) for s in ss]; lens += L; trunc += sum(l >= a.max_tokens for l in L)
    per.append({"question": ex["question"], "gold": ex["answer"], "answers": [extract(s.text) for s in ss], "lens": L,
                "seq": {b: extract(tok.decode(ss[0].token_ids[:b])) for b in budgets}, "first_text": ss[0].text})
rates = [sum(y == x["gold"] for y in x["answers"]) / a.k for x in per]
m = sum(rates) / len(rates)
res = {"tag": a.tag, "model": a.model, "n": len(per), "k_max": a.k, "max_tokens": a.max_tokens,
       "mean_acc": m, "mean_acc_se": (sum((v - m) ** 2 for v in rates) / (len(rates) - 1)) ** 0.5 / len(rates) ** 0.5,
       "maj_at_k": {k: sum(maj(x["answers"][:k]) == x["gold"] for x in per) / len(per) for k in ks},
       "pass_at_k": {k: sum(any(y == x["gold"] for y in x["answers"][:k]) for x in per) / len(per) for k in ks},
       "seq_at_budget": {b: sum(x["seq"][b] == x["gold"] for x in per) / len(per) for b in budgets},
       "mean_len": sum(lens) / len(lens), "frac_truncated": trunc / len(lens), "seconds": round(time.time() - t0, 1),
       "rates": rates, "per": [{k: v for k, v in x.items() if k != "seq"} for x in per]}
json.dump(res, open(a.out + ".tmp", "w")); os.replace(a.out + ".tmp", a.out)
print(json.dumps({k: res[k] for k in ("tag", "n", "mean_acc", "mean_acc_se", "mean_len", "frac_truncated", "seconds")}))
print("maj@k", res["maj_at_k"])

### 0.5 Orchestration helpers

In [ ]:
from common import ks_up_to, extract, maj, SYSTEM

def run(cmd, log_path=None):
    # stream a worker's output into the notebook (and a log file); raise on failure
    cmd = [str(c) for c in cmd]; print(">", " ".join(shlex.quote(c) for c in cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail, f = [], (open(log_path, "a") if log_path else None)
    for line in proc.stdout:
        tail.append(line); tail = tail[-60:]
        if f: f.write(line)
        if line.lstrip().startswith("{") and "'loss'" in line:          # trainer log dict
            kv = dict(re.findall(r"'([^']+)': '?([^',}]+)'?", line))
            print({k: kv[k] for k in ("loss", "mean_token_accuracy", "epoch") if k in kv}, flush=True)
        elif any(t in line for t in ("skipping", "done", "Error", "error", "maj@k", "mean_acc", "training on", "[compat]")):
            print(line.rstrip()[-220:], flush=True)
    proc.wait()
    if f: f.close()
    if proc.returncode != 0:
        print("".join(tail)); raise RuntimeError(f"worker failed (exit {proc.returncode}): {cmd[1]}")

def resolve_teacher(spec):
    """A folder, an archive of one, a Hub repo id, or "latest-full-ft" -> a local folder holding exactly one full model."""
    if spec == "latest-full-ft":
        done = sorted(Path(RESULTS_ROOT, "grpo_full_ft").glob("run-*/derived.json"))
        assert done, f"no finished run under {RESULTS_ROOT}/grpo_full_ft: run the full fine-tuning notebook, or set TEACHER_WEIGHTS to a checkpoint"
        sizes = json.load(open(done[-1]))["sizes"]
        assert "1.5B" in sizes, f"{done[-1].parent.name} has no 1.5B run (it has {list(sizes)})"
        spec = sizes["1.5B"]["final_checkpoint"]; print(f"teacher: {spec} (latest finished full fine-tuning run)")
    p = Path(spec)
    if p.is_file() and p.name.endswith((".zip", ".tar", ".tar.gz", ".tgz")):
        dest = Path("/content/teacher_unpacked")
        if not any(dest.rglob("config.json")):
            dest.mkdir(parents=True, exist_ok=True); shutil.unpack_archive(str(p), str(dest))
        p = dest
    elif not p.exists():
        assert not spec.startswith(("/", ".", "~")), f"{spec} does not exist: upload the checkpoint there or point TEACHER_WEIGHTS at it"
        from huggingface_hub import snapshot_download
        p = Path(snapshot_download(spec))
    assert not list(p.rglob("adapter_config.json")), f"{p} holds a LoRA adapter; the 1.5B GRPO run is full fine-tuning, so this is the wrong folder"
    hits = sorted({c.parent for c in p.rglob("config.json") if list(c.parent.glob("*.safetensors"))})
    assert len(hits) == 1, f"expected one model folder (config.json + *.safetensors) under {p}, found {hits or 'none'}"
    path = hits[0]
    if str(path).startswith("/content/drive"):         # vLLM mmaps weights: read them from local disk, not the Drive FUSE
        local = Path("/content/teacher_grpo")
        if not (local / "config.json").exists():
            shutil.rmtree(local, ignore_errors=True); shutil.copytree(path, local)
        path = local
    return str(path)

def verify_teacher(path, base_id):
    """Same architecture as the base, and not the base itself. Returns a fingerprint of the weights."""
    from huggingface_hub import snapshot_download
    from safetensors import safe_open
    base = snapshot_download(base_id, allow_patterns=["*.json", "*.safetensors", "*.txt"])
    ct, cb = json.load(open(f"{path}/config.json")), json.load(open(f"{base}/config.json"))
    for k in ("model_type", "hidden_size", "num_hidden_layers", "vocab_size"):
        assert ct.get(k) == cb.get(k), f"teacher {k}={ct.get(k)} but {base_id} has {cb.get(k)}: not a fine-tune of {base_id}"
    def tensors(d):
        out = {}
        for f in sorted(Path(d).glob("*.safetensors")):
            with safe_open(str(f), "pt") as h:
                for key in h.keys(): out[key] = (str(f), key)
        return out
    def get(ref):
        with safe_open(ref[0], "pt") as h: return h.get_tensor(ref[1]).float()
    tt, tb = tensors(path), tensors(base)
    keys = sorted(k for k in tt if k in tb and k.endswith("mlp.down_proj.weight"))
    keys = [keys[0], keys[len(keys) // 2], keys[-1]]
    diffs = {k: (get(tt[k]) - get(tb[k])).abs().max().item() for k in keys}
    assert max(diffs.values()) > 0, f"teacher weights equal {base_id}'s on {keys}: this is the base model, not the GRPO checkpoint"
    fp = hashlib.sha256(b"".join(get(tt[k]).numpy().tobytes() for k in keys)).hexdigest()[:16]
    return diffs, fp

def run_eval(tag, model, tokenizer):
    out = RUN / "eval" / f"{tag}.json"
    if not out.exists():
        run([sys.executable, "eval_arm.py", "--model", model, "--tokenizer", tokenizer, "--out", out, "--tag", tag,
             "--n", N_EVAL, "--k", K_EVAL, "--max_tokens", EVAL_MAX_TOKENS, "--budgets", ",".join(map(str, BUDGETS)),
             "--temperature", EVAL_TEMPERATURE, "--seed", SEED], log_path=RUN / "logs" / f"eval_{tag}.log")
    else:
        print(f"[this run, already done] {out.name}")
    return json.load(open(out))

def run_gen(src):
    out = RUN / "harvest" / f"{src}.json"
    if not out.exists():
        s = SOURCES[src]
        run([sys.executable, "gen_traces.py", "--model", s["model"], "--tokenizer", s["tokenizer"], "--out", out,
             "--k", GEN_K, "--max_tokens", GEN_MAX_TOKENS, "--temperature", GEN_TEMPERATURE, "--seed", SEED + 1],
            log_path=RUN / "logs" / f"harvest_{src}.log")
    return json.load(open(out))

# JSON turns int keys into strings; these accessors accept either
getk = lambda r, k: r["maj_at_k"].get(str(k), r["maj_at_k"].get(k))
getp = lambda r, k: r["pass_at_k"].get(str(k), r["pass_at_k"].get(k))
getb = lambda r, b: r["seq_at_budget"].get(str(b), r["seq_at_budget"].get(b))

for t, w in {"so the answer is #### 1,250": "1250", "#### $18.00": "18", "total = 42 so": "42", "none": None, "#### 3\nwait #### 5": "5"}.items():
    assert extract(t) == w, (t, extract(t), w)

# one colour per trace source; a teacher is drawn hollow and dashed in its source's colour
INK, INK2 = "#0b0b0b", "#52514e"
C = {"grpo15": "#2a78d6", "base15": "#eb6834", "self": "#1baf7a", "none": "#8a8984"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#b5b4ae", "axes.grid": True,
                     "grid.color": "#e6e5e1", "grid.linewidth": 0.8, "axes.axisbelow": True, "lines.linewidth": 2,
                     "lines.markersize": 8, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
                     "legend.frameon": False, "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2})
print("helpers ready; extractor OK")

## 1. The teacher, re-measured
The GRPO teacher, its base, and the untrained student, on the GRPO notebook's 600 test problems with its sampling settings.

In [ ]:
TEACHER = resolve_teacher(TEACHER_WEIGHTS)
diffs, TEACHER_FP = verify_teacher(TEACHER, TEACHER_BASE_ID)
fp_file = RUN / "teacher.json"
if fp_file.exists():
    assert json.load(open(fp_file))["fingerprint"] == TEACHER_FP, "this unfinished run was started with different teacher weights"
else:
    json.dump({"weights": TEACHER_WEIGHTS, "local": TEACHER, "fingerprint": TEACHER_FP, "max_abs_diff_vs_base": diffs}, open(fp_file, "w"), indent=1)
print(f"teacher {TEACHER} | fingerprint {TEACHER_FP} | max |Δw| vs base on 3 layers: {[f'{v:.2e}' for v in diffs.values()]}")

SOURCES = {"self":   dict(model=STUDENT_ID,      tokenizer=STUDENT_ID,      label="0.5B itself"),
           "base15": dict(model=TEACHER_BASE_ID, tokenizer=TEACHER_BASE_ID, label="1.5B base"),
           "grpo15": dict(model=TEACHER,         tokenizer=TEACHER_BASE_ID, label="1.5B GRPO")}
ARMS = [("teacher_grpo", "1.5B after GRPO (teacher)", "grpo15", "teacher"),
        ("teacher_base", "1.5B base (teacher)",       "base15", "teacher"),
        ("sft_grpo15",   "0.5B ← GRPO-1.5B traces",   "grpo15", "student"),
        ("sft_base15",   "0.5B ← base-1.5B traces",   "base15", "student"),
        ("sft_self",     "0.5B ← its own traces",     "self",   "student"),
        ("student_base", "0.5B, no SFT",              "none",   "student")]
LABEL = {a: l for a, l, _, _ in ARMS}

EVALS = {"teacher_base": run_eval("teacher_base", TEACHER_BASE_ID, TEACHER_BASE_ID),
         "teacher_grpo": run_eval("teacher_grpo", TEACHER, TEACHER_BASE_ID),
         "student_base": run_eval("student_base", STUDENT_ID, STUDENT_ID)}
display(pd.DataFrame({LABEL[t]: {"pass@1": r["mean_acc"], "±SE": r["mean_acc_se"], f"maj@{K_EVAL}": getk(r, K_EVAL),
                                 f"pass@{K_EVAL}": getp(r, K_EVAL), "mean len": round(r["mean_len"]), "truncated": r["frac_truncated"]}
                      for t, r in EVALS.items()}).T.round(3))

## 2. Harvest
`GEN_K` samples per GSM8K **train** problem from each source, identical sampling. The SFT pool is the problems where **every** source has at least one kept trace.

In [ ]:
HARVEST = {src: run_gen(src) for src in SOURCES}
QS = [r["question"] for r in HARVEST["self"]["rows"]]
assert all([r["question"] for r in HARVEST[s]["rows"]] == QS for s in SOURCES), "sources were sampled on different problem orders"
KEPT = {s: [[(x["text"], x["n_tokens"]) for x in r["samples"] if x["text"] is not None] for r in HARVEST[s]["rows"]] for s in SOURCES}
MATCHED = [i for i in range(len(QS)) if all(KEPT[s][i] for s in SOURCES)]

def harvest_row(s):
    smp = [x for r in HARVEST[s]["rows"] for x in r["samples"]]; kept = [t for ks in KEPT[s] for t in ks]
    return {"train pass@1": np.mean([x["correct"] for x in smp]), "stopped": np.mean([x["stopped"] for x in smp]),
            "problems with a kept trace": np.mean([bool(k) for k in KEPT[s]]), "kept traces": len(kept),
            "kept: mean tokens": round(np.mean([n for _, n in kept])), "kept: writes ####": np.mean(["####" in t for t, _ in kept]),
            "minutes": round(HARVEST[s]["seconds"] / 60, 1)}
HARVEST_TABLE = {SOURCES[s]["label"]: harvest_row(s) for s in SOURCES}
display(pd.DataFrame(HARVEST_TABLE).T.round(3))
print(f"matched pool: {len(MATCHED)} of {len(QS)} train problems ({len(MATCHED)/len(QS):.0%}) — each source trains on one trace per pool problem")

def build_data(src, seed):
    """One kept trace per pool problem, drawn at random (seeded by seed and source). Returns the data path and token counts."""
    path, lens_path = RUN / "data" / f"{src}_seed{seed}.json", RUN / "data" / f"{src}_seed{seed}_lens.json"
    if not path.exists():
        rng = random.Random(f"{seed}:{src}")
        picks = [rng.choice(KEPT[src][i]) for i in MATCHED]
        data = [{"prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": QS[i]}],
                 "completion": [{"role": "assistant", "content": t}]} for i, (t, _) in zip(MATCHED, picks)]
        json.dump(data, open(path, "w")); json.dump([n for _, n in picks], open(lens_path, "w"))
    return path, json.load(open(lens_path))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 1.3]})
srcs = list(SOURCES)
cov = [HARVEST_TABLE[SOURCES[s]["label"]]["problems with a kept trace"] * 100 for s in srcs]
ax[0].barh([SOURCES[s]["label"] for s in srcs], cov, color=[C[s] for s in srcs], height=0.55)
ax[0].axvline(len(MATCHED) / len(QS) * 100, color=INK, lw=1.2, ls=":")
for i, v in enumerate(cov): ax[0].text(v + 1, i, f"{v:.0f}%", va="center", fontsize=9)
ax[0].set(xlim=(0, 108), xlabel="train problems with ≥1 kept trace (%)", title=f"Coverage per source · dotted: matched pool, {len(MATCHED)/len(QS):.0%}"); ax[0].grid(axis="y", visible=False)
bins = np.linspace(0, GEN_MAX_TOKENS, 49)
for s in srcs:
    _, L = build_data(s, SFT_SEEDS[0])
    ax[1].hist(L, bins=bins, histtype="step", lw=2, color=C[s], label=f"{SOURCES[s]['label']} · mean {np.mean(L):.0f}")
ax[1].set(xlabel="tokens per trace", ylabel="traces", title=f"What the student reads (seed {SFT_SEEDS[0]} draw, matched pool)"); ax[1].legend()
plt.tight_layout(); plt.savefig(RUN / "fig_harvest.png", dpi=130); plt.show()

## 3. Matched SFT
One student per source per seed; the seed sets both the trace draw and the data order. Each student is evaluated as soon as it is trained.

In [ ]:
def student(src, seed):
    tag = f"sft_{src}_seed{seed}"
    if (RUN / "eval" / f"{tag}.json").exists() and (RUN / "sft_logs" / f"{tag}.json").exists():
        print(f"[this run, already done] {tag}"); return json.load(open(RUN / "eval" / f"{tag}.json"))
    data, _ = build_data(src, seed); model_dir = LOCAL / tag
    run([sys.executable, "sft_student.py", "--data", data, "--student", STUDENT_ID, "--out", model_dir, "--seed", seed,
         "--epochs", SFT_EPOCHS, "--lr", SFT_LR, "--bsz", SFT_BSZ, "--accum", SFT_ACCUM,
         "--max_len", 256 + GEN_MAX_TOKENS + 64], log_path=RUN / "logs" / f"{tag}.log")
    shutil.copy(model_dir / "log_history.json", RUN / "sft_logs" / f"{tag}.json")
    if src in KEEP_ON_DRIVE and not (RUN / "students" / tag / "config.json").exists():
        shutil.copytree(model_dir, RUN / "students" / tag, dirs_exist_ok=True)
    return run_eval(tag, model_dir, model_dir)

for seed in SFT_SEEDS:                                # seed-major: a disconnect leaves whole seeds behind it
    for src in SOURCES:
        t0 = time.time(); EVALS[f"sft_{src}_seed{seed}"] = student(src, seed)
        print(f"--- sft_{src}_seed{seed}: {(time.time() - t0) / 60:.1f} min")

fig, ax = plt.subplots(figsize=(8, 3.8))
for src in SOURCES:
    for j, seed in enumerate(SFT_SEEDS):
        log = json.load(open(RUN / "sft_logs" / f"sft_{src}_seed{seed}.json"))
        pts = [(r["step"], r["loss"]) for r in log if "loss" in r]
        if pts: ax.plot(*zip(*pts), color=C[src], lw=1.6, alpha=0.9 if j == 0 else 0.35, label=SOURCES[src]["label"] + " traces" if j == 0 else None)
ax.set(xlabel="SFT step", ylabel="loss on the trace (nats/token)", title="Whose reasoning is easiest for 0.5B to imitate"); ax.legend()
plt.tight_layout(); plt.savefig(RUN / "fig_sft_loss.png", dpi=130); plt.show()

## 4. Does the gain travel?
**Left:** pass@1 (mean of 32 samples) per arm; small dots are seeds, bars are 95% intervals. **Right:** paired differences over the same 600 problems.

In [ ]:
golds = [x["gold"] for x in EVALS["teacher_base"]["per"]]
assert all([x["gold"] for x in r["per"]] == golds for r in EVALS.values()), "arms were evaluated on different problems"
IDX = np.random.default_rng(SEED).integers(0, N_EVAL, size=(10_000, N_EVAL))     # one bootstrap, shared by every comparison

def runs_of(arm): return [EVALS[f"{arm}_seed{s}"] for s in SFT_SEEDS] if arm.startswith("sft_") else [EVALS[arm]]
def pooled(arm): return np.mean([r["rates"] for r in runs_of(arm)], axis=0)       # per-problem pass rate, averaged over seeds
def paired(a, b):
    d = pooled(a) - pooled(b); lo, hi = np.percentile(d[IDX].mean(1), [2.5, 97.5]); return d.mean(), lo, hi

GAINS = [("1.5B: GRPO − base",                     "teacher_grpo", "teacher_base", "grpo15"),
         ("0.5B: GRPO traces − base-1.5B traces",  "sft_grpo15",   "sft_base15",   "grpo15"),
         ("0.5B: base-1.5B traces − own traces",   "sft_base15",   "sft_self",     "base15"),
         ("0.5B: own traces − no SFT",             "sft_self",     "student_base", "self"),
         ("0.5B: GRPO traces − no SFT",            "sft_grpo15",   "student_base", "grpo15")]
G = {lab: paired(a, b) for lab, a, b, _ in GAINS}

num, den = pooled("sft_grpo15") - pooled("sft_base15"), pooled("teacher_grpo") - pooled("teacher_base")
nb, db = num[IDX].mean(1), den[IDX].mean(1)
if np.percentile(db, 2.5) > 0:
    TRANSFER = dict(ratio=float(num.mean() / den.mean()), lo=float(np.percentile(nb / db, 2.5)), hi=float(np.percentile(nb / db, 97.5)))
    t_txt = f"transfer of the RL gain: {TRANSFER['ratio']:.0%}  [95% CI {TRANSFER['lo']:.0%}, {TRANSFER['hi']:.0%}]"
else:
    TRANSFER = None; t_txt = "the teacher's GRPO gain is not distinguishable from zero on this run: transfer is undefined"

fig, ax = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1, 1.1]})
for y, (arm, label, src, role) in enumerate(ARMS):
    vals = [r["mean_acc"] * 100 for r in runs_of(arm)]; p = pooled(arm) * 100
    m, half = p.mean(), 1.96 * p.std(ddof=1) / np.sqrt(len(p))
    if len(vals) > 1: ax[0].scatter(vals, [y] * len(vals), s=18, color=C[src], alpha=0.5, zorder=2, lw=0)
    ax[0].errorbar(m, y, xerr=half, fmt="o", ms=9, mfc="white" if role == "teacher" else C[src], mec=C[src], mew=2,
                   ecolor=C[src], elinewidth=2, capsize=0, zorder=3)
    ax[0].text(m, y - 0.3, f"{m:.1f}", ha="center", fontsize=9)
ax[0].set_yticks(range(len(ARMS))); ax[0].set_yticklabels([l for _, l, _, _ in ARMS]); ax[0].invert_yaxis(); ax[0].grid(axis="y", visible=False)
ax[0].set(xlabel=f"GSM8K pass@1, % (mean of {K_EVAL} samples, n={N_EVAL})", title="Every arm on the same 600 problems")
for y, (lab, a, b, src) in enumerate(GAINS):
    d, lo, hi = (v * 100 for v in G[lab]); headline = y in (0, 1)
    col = C[src] if headline else INK2
    ax[1].errorbar(d, y, xerr=[[d - lo], [hi - d]], fmt="o", ms=9, color=col, mfc=col if headline else "white", mew=2, elinewidth=2, capsize=0)
    ax[1].text(hi + 0.4, y, f"{d:+.1f}  [{lo:+.1f}, {hi:+.1f}]", va="center", fontsize=9)
ax[1].axvline(0, color=INK, lw=1)
ax[1].set_yticks(range(len(GAINS))); ax[1].set_yticklabels([g[0] for g in GAINS]); ax[1].invert_yaxis(); ax[1].grid(axis="y", visible=False)
xmax = max(G[g][2] for g in G) * 100
ax[1].set(xlim=(min(0, min(G[g][1] for g in G) * 100) - 1, xmax + 9), xlabel="paired difference in pass@1, points (95% bootstrap CI)", title=t_txt)
plt.tight_layout(); plt.savefig(RUN / "fig_transfer.png", dpi=130, bbox_inches="tight"); plt.show()

def summary(arm):
    rs, p = runs_of(arm), pooled(arm)
    return {"pass@1": p.mean(), "±SE": p.std(ddof=1) / np.sqrt(len(p)),
            "seed range": (max(r["mean_acc"] for r in rs) - min(r["mean_acc"] for r in rs)) if len(rs) > 1 else np.nan,
            "maj@1": np.mean([getk(r, 1) for r in rs]), f"maj@{K_EVAL}": np.mean([getk(r, K_EVAL) for r in rs]),
            f"pass@{K_EVAL}": np.mean([getp(r, K_EVAL) for r in rs]), "mean len": round(np.mean([r["mean_len"] for r in rs])),
            "truncated": np.mean([r["frac_truncated"] for r in rs])}
SUMMARY = {LABEL[a]: summary(a) for a, _, _, _ in ARMS}
display(pd.DataFrame(SUMMARY).T.round(3))
per_seed = {s: (EVALS[f"sft_grpo15_seed{s}"]["mean_acc"] - EVALS[f"sft_base15_seed{s}"]["mean_acc"]) * 100 for s in SFT_SEEDS}
print("per seed, GRPO traces − base-1.5B traces (points):", {s: round(v, 1) for s, v in per_seed.items()})
for a, r in SUMMARY.items():
    if r["truncated"] > 0.05: print(f"FLAG {a}: {r['truncated']:.1%} of samples hit {EVAL_MAX_TOKENS} tokens; its curve is capped by the budget, not the model")

## 5. Both axes for the student
The GRPO notebook's two test-time readings, per arm. Teachers dashed and hollow; SFT arms averaged over seeds.

In [ ]:
ks = ks_up_to(K_EVAL)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4), gridspec_kw={"width_ratios": [1, 1, 0.9]})
for arm, label, src, role in ARMS:
    rs = runs_of(arm); sty = dict(color=C[src], ls="--" if role == "teacher" else "-", marker="o", mfc="white" if role == "teacher" else C[src], mew=2, label=label)
    ax[0].plot(ks, [np.mean([getk(r, k) for r in rs]) * 100 for k in ks], **sty)
    ax[1].plot(BUDGETS, [np.mean([getb(r, b) for r in rs]) * 100 for b in BUDGETS], **sty)
for a, xs, xl, tl in [(ax[0], ks, "samples k, majority vote (log)", "Parallel: maj@k"),
                      (ax[1], BUDGETS, "thinking budget, tokens (log)", "Sequential: one sample, cut at a budget")]:
    a.set_xscale("log", base=2); a.set_xticks(xs); a.set_xticklabels(xs); a.set(xlabel=xl, ylabel="GSM8K accuracy (%)", title=tl)
fig.legend(*ax[0].get_legend_handles_labels(), loc="lower center", ncol=len(ARMS), bbox_to_anchor=(0.5, -0.07), fontsize=9)
ys = range(len(ARMS)); L = [np.mean([r["mean_len"] for r in runs_of(a)]) for a, _, _, _ in ARMS]
ax[2].barh(ys, L, height=0.6, color=["white" if role == "teacher" else C[src] for _, _, src, role in ARMS],
           edgecolor=[C[src] for _, _, src, _ in ARMS], linewidth=2)
for y, v in zip(ys, L): ax[2].text(v + 4, y, f"{v:.0f}", va="center", fontsize=9)
ax[2].set_yticks(list(ys)); ax[2].set_yticklabels([l for _, l, _, _ in ARMS]); ax[2].invert_yaxis(); ax[2].grid(axis="y", visible=False)
ax[2].set(xlabel="mean tokens per test sample", title="Does the student inherit the length?", xlim=(0, max(L) * 1.18))
plt.tight_layout(); plt.savefig(RUN / "fig_student_axes.png", dpi=130, bbox_inches="tight"); plt.show()

## 6. One problem, three models
A test problem drawn at random (seeded) from all 600; each model's first sample, graded by the same extractor.

In [ ]:
i = random.Random(f"specimen:{SEED}").randrange(N_EVAL); s0 = SFT_SEEDS[0]
x0 = EVALS["teacher_grpo"]["per"][i]
print(f"test problem {i} | gold {x0['gold']}\n{x0['question']}\n")
for tag, lab in [("teacher_grpo", LABEL["teacher_grpo"]), (f"sft_base15_seed{s0}", LABEL["sft_base15"]), (f"sft_grpo15_seed{s0}", LABEL["sft_grpo15"])]:
    x = EVALS[tag]["per"][i]; a = x["answers"][0]
    print(f"===== {lab} | answer {a} ({'correct' if a == x['gold'] else 'wrong'}) | {x['lens'][0]} tokens | "
          f"this problem: {sum(y == x['gold'] for y in x['answers'])}/{K_EVAL} samples correct\n{x['first_text']}\n")

## 7. Record the run
Writes `derived.json`, which marks the run finished: re-running the notebook after this starts a new run.

In [ ]:
def vit(r): return r.get("seconds")
derived = {
    "finished_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "knobs": KNOBS,
    "env": {"gpu": gpu.name, "torch": torch.__version__, "vllm": vllm.__version__, "trl": trl.__version__, "transformers": transformers.__version__},
    "teacher": json.load(open(RUN / "teacher.json")),
    "harvest": {s: HARVEST_TABLE[SOURCES[s]["label"]] for s in SOURCES}, "n_train_problems": len(QS), "n_matched": len(MATCHED),
    "arms": {a: {"label": LABEL[a], "source": src, "role": role, "summary": SUMMARY[LABEL[a]],
                 "per_seed_mean_acc": [r["mean_acc"] for r in runs_of(a)],
                 "maj_at_k": {k: float(np.mean([getk(r, k) for r in runs_of(a)])) for k in ks},
                 "seq_at_budget": {b: float(np.mean([getb(r, b) for r in runs_of(a)])) for b in BUDGETS},
                 "rates": pooled(a).round(5).tolist()} for a, _, src, role in ARMS},
    "gains_points": {lab: [round(v * 100, 2) for v in G[lab]] for lab in G}, "transfer": TRANSFER,
    "seconds": {"harvest": {s: HARVEST[s]["seconds"] for s in SOURCES}, "eval": {t: vit(r) for t, r in EVALS.items()},
                "sft": {f"sft_{s}_seed{sd}": next((r.get("train_runtime") for r in json.load(open(RUN / "sft_logs" / f"sft_{s}_seed{sd}.json")) if "train_runtime" in r), None)
                        for s in SOURCES for sd in SFT_SEEDS}},
}
derived = json.loads(json.dumps(derived, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
json.dump(derived, open(RUN / "derived.json.tmp", "w"), indent=1); os.replace(RUN / "derived.json.tmp", RUN / "derived.json")
print(f"{RUN}/derived.json | {t_txt}")
print("students kept on Drive:", sorted(p.name for p in (RUN / "students").iterdir()) or "none")

## Reading the results
- **Transfer near 100%** means the GRPO gain is carried by the traces; **near 0%** means the 0.5B learns as much from the base 1.5B's traces as from the trained ones, and the gain lives in what the teacher does with its own weights.
- **Own traces vs. base-1.5B traces** separates "a bigger model wrote it" from "any correct GSM8K trace helps". If own traces match the 1.5B's, the student is learning the task format, not the teacher.
- **Seed spread.** The paired CI covers sampling noise over problems, not SFT variance; read the seed dots in §4 and the per-seed line. A gap smaller than the seed range is not a finding.
- **Matching trims hard problems.** The pool requires a kept trace from the 0.5B too, so problems only the GRPO teacher solves are excluded. The harvest table says how much that costs.

### Troubleshooting
| Symptom | Fix |
|---|---|
| `... does not exist: upload the checkpoint there` | Copy `step400` (the folder with `config.json` + `model.safetensors`) to `TEACHER_WEIGHTS`, or point it at a `.zip` / Hub repo id |
| a Drive download arrives as `step400-…-002.zip`, `-003.zip` and a loose `model-001.safetensors` | Drive splits big folders. Easiest: point `TEACHER_WEIGHTS` at the folder on Drive, which is whole. Otherwise put `config.json`, `generation_config.json` and the safetensors (renamed `model.safetensors`) in one folder and point at it |
| `holds a LoRA adapter` / `equal ... base model` | wrong folder: the 1.5B GRPO checkpoint is a full model and must differ from the base |
| `unfinished but was started with other knobs` | restore the knobs, or rename that `run-*` folder to start fresh |
| `[compat] SFTConfig does not accept ...` | a TRL rename; `completion_only_loss` is TRL's default for prompt-completion data anyway |
| SFT OOM | halve `SFT_BSZ` and double `SFT_ACCUM` (effective batch unchanged, so the run's knobs change: start a new run) |
| install / NCCL errors | see the GRPO notebook's troubleshooting table; the install cell is identical |